# Module 3: Choosing a Denominator

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A count of use of force incidents tells you how much happened. It cannot tell
you whether an agency uses force often, because a busy agency will always record
more of everything.

To answer that you need a denominator. There are four sensible ones, they
disagree, and choosing between them is not a technical question. It is a
question about what you are asking.

**About 15 minutes.**

## 1. Build the 2023 numbers

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

y23 = (monthly[monthly["year_month"].str[:4] == "2023"]
       .groupby("agency_id", as_index=False)
       .agg(uof=("n_uof", "sum"), arrests=("n_arrests", "sum"),
            calls=("total_cfs", "sum"), officers=("sworn_officers", "max")))

y23 = y23.merge(profile[["agency_id", "agency_name", "population_served"]],
                on="agency_id")
y23["agency"] = (y23["agency_name"].str.replace(" Police Department", "", regex=False)
                                   .str.replace(" Sheriff's Office", "", regex=False)
                                   .str.replace(" Police", "", regex=False))
y23[["agency", "uof", "arrests", "calls", "officers", "population_served"]]

## 2. The four denominators

Each one is a different question:

| Rate | The question it answers |
|---|---|
| per 100 arrests | when officers take someone into custody, how often is force involved |
| per 1,000 calls for service | across everything officers are sent to, how often does force result |
| per 1,000 residents | how much force does this community experience |
| per sworn officer | how much force does the average officer in this agency apply |

In [ ]:
y23["per_100_arrests"] = 100 * y23["uof"] / y23["arrests"]
y23["per_1000_calls"] = 1000 * y23["uof"] / y23["calls"]
y23["per_1000_residents"] = 1000 * y23["uof"] / y23["population_served"]
y23["per_officer"] = y23["uof"] / y23["officers"]

rates = ["per_100_arrests", "per_1000_calls", "per_1000_residents", "per_officer"]
y23.set_index("agency")[rates].round(2).sort_values("per_100_arrests", ascending=False)

## 3. The denominator decides the ranking

Rank each agency under each rate and compare.

In [ ]:
ranks = y23.set_index("agency")[rates].rank(ascending=False).astype(int)
ranks["biggest gap"] = ranks.max(axis=1) - ranks.min(axis=1)
ranks.sort_values("biggest gap", ascending=False)

Two agencies move a long way.

**Pinecrest State University** is fifth by arrests and eleventh by residents.
Its 29,000 residents are students, most of whom leave in June, and the campus
also serves visitors who are not in that count. The denominator does not mean
the same thing for a campus force as it does for a city.

**Orrindale** is fourth by arrests and first by residents. It is an eight officer
department with 3,900 residents and fourteen incidents in the whole year.
Whether that ranking means anything at all is the subject of Module 4.

## 4. A denominator can be part of what you are measuring

Arrests are the most natural denominator for use of force, and they carry a
problem: an agency can change its arrest rate. If a department arrests fewer
people but uses force in the same situations, its rate per arrest rises without
any change in officer behaviour.

Calls for service are less exposed to this, because they are generated by the
public rather than chosen by the agency. Neither is clean. Reporting both is the
honest answer.

In [ ]:
compare = y23.set_index("agency")[["per_100_arrests", "per_1000_calls"]].round(2)
compare["rank by arrests"] = compare["per_100_arrests"].rank(ascending=False).astype(int)
compare["rank by calls"] = compare["per_1000_calls"].rank(ascending=False).astype(int)
compare.sort_values("rank by arrests")

## 5. The coverage trap

The numerator and the denominator must cover the same months. Prairie County
never submitted three months of 2022, so both its incidents and its arrests for
that year are nine month figures. An annual rate is still correct, because both
sides are short by the same three months. An annual **count** compared against
other agencies is not.

In [ ]:
pc22 = monthly[(monthly["agency_id"] == "A009") &
               (monthly["year_month"].str[:4] == "2022")]

print(f"months of 2022 present: {len(pc22)}")
print(f"incidents: {pc22['n_uof'].sum()}   arrests: {pc22['n_arrests'].sum():,}")
print(f"rate per 100 arrests: {100 * pc22['n_uof'].sum() / pc22['n_arrests'].sum():.2f}"
      "   <- fine, both sides cover the same nine months")
print(f"annual count: {pc22['n_uof'].sum()}"
      "   <- not comparable to a twelve month agency")

## 6. The denominator moves too

Everything so far has compared agencies at one moment. A rate also has a
**trend**, and because a rate is a ratio, its trend is the numerator's trend
minus the denominator's:

> change in the rate  **=**  change in the count  **minus**  change in the denominator

That matters more than it sounds. If arrests are growing, every agency's rate
improves before anything about officer behaviour changes.

In [ ]:
import statsmodels.formula.api as smf

per_year = lambda b: 100 * (np.exp(12 * b) - 1)

clean = monthly[(monthly["provisional"] == 0) & (monthly["year_month"] <= "2025-12")]
clean = clean[~((clean["agency_id"] == "A002") & (clean["year_month"] == "2021-06"))]

rows = []
for aid, g in clean.groupby("agency_id"):
    g = g.sort_values("year_month").copy()
    g["t"] = np.arange(len(g))
    g["rate"] = 100 * g["n_uof"] / g["n_arrests"]
    rows.append({
        "agency": g["agency_name"].iloc[0],
        "count": per_year(smf.ols("np.log(n_uof + 0.5) ~ t", data=g).fit().params["t"]),
        "arrests": per_year(smf.ols("np.log(n_arrests) ~ t", data=g).fit().params["t"]),
        "rate": per_year(smf.ols("np.log(rate) ~ t", data=g[g["rate"] > 0]).fit().params["t"]),
    })

trends = pd.DataFrame(rows).set_index("agency").round(2).sort_values("rate")
trends

Arrests grew between about 1.2 and 1.9 percent a year almost everywhere, so
every agency's rate falls by roughly that much for free.

Two rows deserve a second look.

In [ ]:
trends.loc[["Dunmoor Tribal Police", "Havenbrook Police Department"]]

**Dunmoor Tribal recorded more incidents each year and its rate was flat.**
A report on counts calls that a deterioration. A report on rates calls it no
change. Both are correct, and they lead to opposite decisions.

**Havenbrook's rate improved faster than its incidents fell.** More than a
third of the apparent improvement is arrests going up rather than incidents
coming down.

So: whenever you report a trend in a rate, report the denominator's trend beside
it. Otherwise nobody can tell which half of the ratio moved.

In [ ]:
check = trends.copy()
check["count minus arrests"] = (check["count"] - check["arrests"]).round(2)
check["gap from the rate"] = (check["rate"] - check["count minus arrests"]).round(2)
check[["count", "arrests", "rate", "count minus arrests", "gap from the rate"]]

The identity holds closely for most agencies. It loosens where counts are very
small and a month of zero has to be offset before taking logs, which is why
Prairie County is the visible exception. Approximations that depend on sample
size keep reappearing, and
[Module 4](Module_04_Why_Small_Agencies_Look_Volatile.ipynb) is about that.

## 7. A reusable helper

In [ ]:
def agency_rates(monthly, profile, year):
    """Four standard rates per agency for one calendar year."""
    a = (monthly[(monthly["year_month"].str[:4] == str(year)) &
                 (monthly["provisional"] == 0)]
         .groupby("agency_id", as_index=False)
         .agg(uof=("n_uof", "sum"), arrests=("n_arrests", "sum"),
              calls=("total_cfs", "sum"), officers=("sworn_officers", "max"),
              months=("year_month", "nunique")))
    a = a.merge(profile[["agency_id", "agency_name", "population_served"]], on="agency_id")
    a["per_100_arrests"] = 100 * a["uof"] / a["arrests"]
    a["per_1000_calls"] = 1000 * a["uof"] / a["calls"]
    a["per_1000_residents"] = 1000 * a["uof"] / a["population_served"]
    a["per_officer"] = a["uof"] / a["officers"]
    return a

In [ ]:
r = agency_rates(monthly, profile, 2025)
r[["agency_name", "months", "uof", "per_100_arrests", "per_1000_residents"]].round(2)

The `months` column is there on purpose. Any agency showing fewer than twelve
had a coverage problem that year, and its counts should not be compared
directly against the others.

## 8. What to carry away

- **Name the denominator every time.** "The rate rose" is not a statement.
- **Report at least two.** If they agree the finding is robust. If they disagree
  that is itself the finding.
- **Check that both sides cover the same months.**
- **Report the denominator's own trend** next to any trend in a rate.
- **Ask whether the denominator can move on its own.** Arrests can. Population
  mostly cannot.

## Exercise

Find the agency whose rank changes the most between per 100 arrests and per
1,000 residents in 2025, and work out why.

In [ ]:
# Fill in the blanks, then run.
YEAR = None                # try 2025
LEFT, RIGHT = None, None   # try "per_100_arrests" and "per_1000_residents"

if YEAR and LEFT and RIGHT:
    r = agency_rates(monthly, profile, YEAR).set_index("agency_name")
    out = pd.DataFrame({
        LEFT: r[LEFT].rank(ascending=False).astype(int),
        RIGHT: r[RIGHT].rank(ascending=False).astype(int)})
    out["move"] = (out[LEFT] - out[RIGHT]).abs()
    print(out.sort_values("move", ascending=False).to_string())
else:
    print("Set YEAR, LEFT and RIGHT above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
YEAR = 2025
LEFT, RIGHT = "per_100_arrests", "per_1000_residents"
```

Pinecrest State University moves furthest, from fourth by arrests to eleventh
by residents, for the reason given in section 3: its residents are students, and
a student population is not comparable to a city population as a measure of who
is exposed to policing.

Kelsmoor and Dunmoor Tribal each move five places, in opposite directions.
Both are small, and Module 4 shows that rankings of small agencies are largely
an artifact of how few incidents they record.

None of these agencies changed. Only the question did.

</details>

---

**Next:** [Module 4, Why Small Agencies Look Volatile](Module_04_Why_Small_Agencies_Look_Volatile.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*